<a href="https://colab.research.google.com/github/RegNLP/RePASs/blob/main/RePASsL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install textstat wordfreq hf_xet

In [ ]:
# !pip install sentence-transformers

In [ ]:
import os
import json
import torch
import random
import numpy as np
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Set up random seeds and deterministic flags for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class ObligationClassifierTrainer:
    def __init__(self, data_path: str, save_path: str):
        self.data_path = data_path
        self.save_path = save_path
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Set seeds for reproducibility
        set_seed(42)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        print(f"Using device: {self.device}")
        os.makedirs(self.save_path, exist_ok=True)

    def load_data(self):
        with open(self.data_path, 'r') as f:
            data = json.load(f)
        self.texts = [item['Text'] for item in data]
        self.labels = [1 if item['Obligation'] else 0 for item in data]

    class ObligationDataset(Dataset):
        def __init__(self, texts, labels, tokenizer, max_len=128):
            self.texts = texts
            self.labels = labels
            self.tokenizer = tokenizer
            self.max_len = max_len

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            text = self.texts[idx]
            label = self.labels[idx]
            encoding = self.tokenizer.encode_plus(
                text,
                add_special_tokens=True,
                max_length=self.max_len,
                return_token_type_ids=False,
                truncation=True,
                padding='max_length',
                return_attention_mask=True,
                return_tensors='pt',
            )
            return {
                'text': text,
                'input_ids': encoding['input_ids'].flatten(),
                'attention_mask': encoding['attention_mask'].flatten(),
                'labels': torch.tensor(label, dtype=torch.long)
            }

    def compute_metrics(self, pred):
        labels = pred.label_ids
        preds = pred.predictions.argmax(-1)
        precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
        acc = accuracy_score(labels, preds)
        return {
            'accuracy': acc,
            'f1': f1,
            'precision': precision,
            'recall': recall
        }

    def train(self):
        print("📦 Loading and preparing data...")
        self.load_data()
        tokenizer = AutoTokenizer.from_pretrained('nlpaueb/legal-bert-base-uncased')

        X_train, X_val, y_train, y_val = train_test_split(
            self.texts, self.labels, test_size=0.2, random_state=42
        )
        train_dataset = self.ObligationDataset(X_train, y_train, tokenizer)
        val_dataset = self.ObligationDataset(X_val, y_val, tokenizer)

        model = AutoModelForSequenceClassification.from_pretrained(
            'nlpaueb/legal-bert-base-uncased', num_labels=2
        )
        model.to(self.device)

        # Create directories for logs and output
        output_dir = os.path.join(self.save_path, 'results')
        log_dir = os.path.join(self.save_path, 'logs')
        os.makedirs(output_dir, exist_ok=True)
        os.makedirs(log_dir, exist_ok=True)

        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=10,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            warmup_steps=500,
            weight_decay=0.01,
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            logging_dir=log_dir,
            logging_steps=10,
            seed=42
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=self.compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
        )

        print("🚀 Starting training...")
        trainer.train()

        print("📊 Evaluating model...")
        eval_results = trainer.evaluate()
        print(f"Evaluation results: {eval_results}")

        # === ✅ FIX: Manually save best model checkpoint to classifier folder ===
        best_model_path = trainer.state.best_model_checkpoint
        print("💾 Saving best model manually from checkpoint:", best_model_path)

        best_model = AutoModelForSequenceClassification.from_pretrained(best_model_path)
        best_model.save_pretrained(self.save_path)
        tokenizer.save_pretrained(self.save_path)

        print("✅ Model successfully saved to:", self.save_path)
        print("📂 Files in classifier folder:", os.listdir(self.save_path))


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
import numpy as np
from nltk.tokenize import sent_tokenize

class RePASsScorer:
    def __init__(self, obligation_model_path: str, device=None):
        self.device = torch.device(device if device else ("cuda" if torch.cuda.is_available() else "cpu"))
        self.obligation_model_path = obligation_model_path

        # Load obligation classifier
        self.obligation_tokenizer = AutoTokenizer.from_pretrained(obligation_model_path)
        self.obligation_model = AutoModelForSequenceClassification.from_pretrained(obligation_model_path).to(self.device).eval()

        # Load NLI models
        self.nli_tokenizer = AutoTokenizer.from_pretrained('cross-encoder/nli-deberta-v3-xsmall')
        self.nli_model = AutoModelForSequenceClassification.from_pretrained('cross-encoder/nli-deberta-v3-xsmall').to(self.device).eval()

        self.coverage_nli_model = pipeline(
            "text-classification", model="microsoft/deberta-large-mnli",
            device=0 if torch.cuda.is_available() else -1
        )

    def _softmax(self, logits):
        e_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
        return e_logits / np.sum(e_logits, axis=1, keepdims=True)

    def _get_nli_matrix(self, premises, hypotheses):
        entailment_matrix = np.zeros((len(premises), len(hypotheses)))
        contradiction_matrix = np.zeros((len(premises), len(hypotheses)))
        batch_size = 16
        for i, p in enumerate(premises):
            for j in range(0, len(hypotheses), batch_size):
                h_batch = hypotheses[j:j+batch_size]
                features = self.nli_tokenizer([p]*len(h_batch), h_batch, padding=True, truncation=True, return_tensors="pt").to(self.device)
                with torch.no_grad():
                    logits = self.nli_model(**features).logits.cpu().numpy()
                probs = self._softmax(logits)
                entailment_matrix[i, j:j+len(h_batch)] = probs[:, 1]
                contradiction_matrix[i, j:j+len(h_batch)] = probs[:, 0]
        return entailment_matrix, contradiction_matrix

    def _score_matrix(self, matrix: np.ndarray) -> float:
        return round(np.mean(np.max(matrix, axis=0)), 5) if matrix.size > 0 else 0.0

    def _classify_obligations(self, sentences):
        inputs = self.obligation_tokenizer(sentences, padding=True, truncation=True, return_tensors='pt').to(self.device)
        with torch.no_grad():
            logits = self.obligation_model(**inputs).logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        return [s for s, p in zip(sentences, preds) if p == 1]

    def _obligation_coverage(self, passage_sentences, answer_sentences):
        source_obligations = self._classify_obligations(passage_sentences)
        answer_obligations = self._classify_obligations(answer_sentences)

        if not source_obligations:
            return 0.0

        covered = 0
        for rule in source_obligations:
            for ans in answer_obligations:
                result = self.coverage_nli_model(f"{ans} [SEP] {rule}")
                if result[0]['label'].lower() == 'entailment' and result[0]['score'] > 0.7:
                    covered += 1
                    break

        return covered / len(source_obligations)

    def score(self, passage: str, answer: str) -> dict:
        passage_sents = sent_tokenize(passage)
        answer_sents = sent_tokenize(answer)

        entailment_matrix, contradiction_matrix = self._get_nli_matrix(passage_sents, answer_sents)
        entailment_score = self._score_matrix(entailment_matrix)
        contradiction_score = self._score_matrix(contradiction_matrix)
        obligation_coverage_score = self._obligation_coverage(passage_sents, answer_sents)

        repass_score = round((obligation_coverage_score + entailment_score - contradiction_score + 1) / 3, 5)

        return {
            "entailment_score": entailment_score,
            "contradiction_score": contradiction_score,
            "obligation_coverage_score": round(obligation_coverage_score, 5),
            "repass_score": repass_score
        }


In [ ]:
import torch
import numpy as np
import nltk
import textstat
from wordfreq import zipf_frequency
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from torch.nn.utils.rnn import pad_sequence

nltk.download('stopwords')
nltk.download('punkt')

# ---------------------- Fluency Score ----------------------
class FluencyScore:
    def __init__(self, device=None, same_length=False):
        self.device = torch.device(device if device else ("cuda" if torch.cuda.is_available() else "cpu"))
        self.model = GPT2LMHeadModel.from_pretrained("gpt2").to(self.device)
        self.tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
        if self.device.type == "cuda":
            self.model = self.model.half()
        self.model.eval()
        self.same_length = same_length
        self.max_output_length = 80

    def split_into_chunks(self, text):
        if not text or not isinstance(text, str):
            return []
        tokens = self.tokenizer.encode(text)
        return [tokens[i:i + (self.max_output_length - 1)] for i in range(0, len(tokens), self.max_output_length - 1)]

    def preprocess_batch(self, decoded):
        all_chunks = []
        for dec in decoded:
            chunks = self.split_into_chunks(dec)
            if chunks:
                all_chunks.extend(chunks)
        if not all_chunks:
            return None, None, 0

        try:
            decs_inp = pad_sequence(
                [torch.LongTensor([self.tokenizer.bos_token_id] + chunk) for chunk in all_chunks],
                batch_first=True, padding_value=0
            ).to(self.device)

            decs_out = pad_sequence(
                [torch.LongTensor(chunk + [self.tokenizer.eos_token_id]) for chunk in all_chunks],
                batch_first=True, padding_value=-1
            ).to(self.device)

            return decs_inp, decs_out, len(all_chunks)
        except Exception as e:
            print(f"Error in batch preprocessing: {str(e)}")
            return None, None, 0

    def text2loss(self, text_list):
        txt_inp, txt_out, num_chunks = self.preprocess_batch(text_list)
        if num_chunks == 0 or txt_inp is None or txt_out is None:
            return float('inf')
        try:
            with torch.no_grad():
                model_outputs = self.model(input_ids=txt_inp)
                crit = torch.nn.CrossEntropyLoss(ignore_index=-1, reduction='none')
                loss = crit(model_outputs["logits"].view(-1, self.tokenizer.vocab_size),
                            txt_out.view(-1)).view(txt_out.shape)

                mask = (txt_inp != 0).float()
                non_pad_count = torch.sum(mask, dim=1)
                loss_per_chunk = torch.sum(loss, dim=1) / non_pad_count

            return loss_per_chunk.mean().item()
        except Exception as e:
            print(f"Error calculating text loss: {str(e)}")
            return float('inf')

    def score(self, sources, generateds, printing=False):
        if not sources or not generateds or len(sources) != len(generateds):
            return {"scores": [], "sources_loss": [], "generateds_loss": []}

        sources_score = torch.tensor([self.text2loss([src]) for src in sources])
        generateds_score = torch.tensor([self.text2loss([gen]) for gen in generateds])

        scores = (1.3 + sources_score - generateds_score) / 1.3
        scores = torch.clamp(scores, 0.001, 1.0).tolist()

        return {
            "scores": scores,
            "sources_loss": sources_score.tolist(),
            "generateds_loss": generateds_score.tolist()
        }

# ---------------------- Readability Score ----------------------
class ReadabilityScore:
    def compute(self, passage, answer):
        if not passage or not answer:
            return {
                "fkgl": (0, 0),
                "fre": (0, 0),
                "readability_score": (0, 0)
            }

        try:
            fkgl_p = textstat.flesch_kincaid_grade(passage)
            fre_p = textstat.flesch_reading_ease(passage)
            fkgl_a = textstat.flesch_kincaid_grade(answer)
            fre_a = textstat.flesch_reading_ease(answer)

            norm_fkgl_p = max(0.0, min(1.0, 1 - fkgl_p / 18))
            norm_fkgl_a = max(0.0, min(1.0, 1 - fkgl_a / 18))
            norm_fre_p = max(0.0, min(1.0, fre_p / 100))
            norm_fre_a = max(0.0, min(1.0, fre_a / 100))

            combined_p = round((norm_fkgl_p + norm_fre_p) / 2, 5)
            combined_a = round((norm_fkgl_a + norm_fre_a) / 2, 5)

            return {
                "fkgl": (fkgl_p, fkgl_a),
                "fre": (fre_p, fre_a),
                "readability_score": (combined_p, combined_a)
            }
        except Exception as e:
            print(f"Error calculating readability: {str(e)}")
            return {
                "fkgl": (0, 0),
                "fre": (0, 0),
                "readability_score": (0, 0)
            }

# ---------------------- Simplicity Score ----------------------
def shift_to_score(shift, target_shift, right_slope=0.25):
    if shift <= target_shift:
        score = shift / (target_shift + 0.001)
    else:
        score = 1.0 - right_slope * (shift - target_shift) / (target_shift + 0.001)
    return np.clip(score, 0, 1.0)

class SimplicityScore:
    def __init__(self, max_grade=30, target_lexical_shift=0.4):
        self.max_grade = max_grade
        self.target_lexical_shift = target_lexical_shift
        self.stopws = set(nltk.corpus.stopwords.words("english") + ["might", "would", "``"])

    def is_good_word(self, w):
        return len(w) > 1 and len(w) < 30 and "'" not in w and not all(c.isdigit() for c in w) and w.lower() not in self.stopws

    def word_score_func(self, w):
        return zipf_frequency(w, 'en', wordlist="large")

    def compute_lexical_zipf(self, text):
        try:
            words = nltk.tokenize.word_tokenize(text)
            good_words = [w.lower() for w in words if self.is_good_word(w)]
            if not good_words:
                return 0.0
            return np.mean([self.word_score_func(w) for w in good_words])
        except:
            return 0.0

    def lexical_score(self, passage, answer):
        zipf_source = self.compute_lexical_zipf(passage)
        zipf_generated = self.compute_lexical_zipf(answer)
        shift = zipf_generated - zipf_source
        score = shift_to_score(shift, self.target_lexical_shift)
        return zipf_source, zipf_generated, shift, score

    def syntactic_score(self, passage, answer):
        try:
            fkgl_source = textstat.flesch_kincaid_grade(passage)
            fkgl_generated = textstat.flesch_kincaid_grade(answer)
            rshift = fkgl_source - fkgl_generated
            if fkgl_source <= 4.0:
                target_shift = 0
            elif fkgl_source <= 12.0:
                target_shift = (fkgl_source - 3) * 0.5
            else:
                target_shift = 4.5 + (fkgl_source - 12) * 0.83

            score = shift_to_score(rshift, target_shift)
            return fkgl_source, fkgl_generated, rshift, score
        except:
            return 0, 0, 0, 0

    def compute(self, passage, answer):
        zipf_source, zipf_gen, zipf_shift, lex_score = self.lexical_score(passage, answer)
        fkgl_source, fkgl_gen, fkgl_shift, syn_score = self.syntactic_score(passage, answer)
        final_score = round((lex_score + syn_score) / 2, 5)
        return {
            "source_zipf": zipf_source,
            "generated_zipf": zipf_gen,
            "zipf_shift": zipf_shift,
            "lexical_score": lex_score,
            "source_fkgl": fkgl_source,
            "generated_fkgl": fkgl_gen,
            "fkgl_shift": fkgl_shift,
            "syntactic_score": syn_score,
            "simplicity_score": final_score
        }

# ---------------------- Unified Linguistic Scorer ----------------------
class LinguisticScorer:
    def __init__(self, device=None):
        self.fluency = FluencyScore(device=device)
        self.readability = ReadabilityScore()
        self.simplicity = SimplicityScore()

    def score(self, passage: str, answer: str) -> dict:
        if not passage or not answer:
            return {
                "fluency_score": 0.0,
                "readability_score": 0.0,
                "simplicity_score": 0.0,
                "combined_linguistic_score": 0.0
            }

        try:
            flu = self.fluency.score([passage], [answer])
            fluency_score = round(flu["scores"][0], 5) if flu["scores"] else 0.0
            flu_loss_gen = flu["generateds_loss"][0] if flu["generateds_loss"] else float('inf')
            flu_loss_score = 1 / (1 + flu_loss_gen) if flu_loss_gen != float('inf') else 0.0

            readability_result = self.readability.compute(passage, answer)
            readability_score = round(readability_result["readability_score"][1], 5)

            simplicity_result = self.simplicity.compute(passage, answer)
            simplicity_score = round(simplicity_result["simplicity_score"], 5)

            combined = round((flu_loss_score + readability_score + simplicity_score) / 3, 5)

            return {
                "fluency_score": fluency_score,
                "readability_score": readability_score,
                "simplicity_score": simplicity_score,
                "combined_linguistic_score": combined
            }

        except Exception as e:
            print(f"[LinguisticScorer] Error: {e}")
            return {
                "fluency_score": 0.0,
                "readability_score": 0.0,
                "simplicity_score": 0.0,
                "combined_linguistic_score": 0.0
            }


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
import os
import json
import csv
import pandas as pd
import nltk
nltk.download('punkt_tab')

class RePASSLRunner:
    def __init__(self, folder_path: str, input_file: str, output_file: str, classifier_data_file: str = None):
        self.folder_path = folder_path
        self.input_file = input_file
        self.output_file = output_file
        self.classifier_path = os.path.join(folder_path, "obligation-classifier-legalbert")
        self.classifier_data_file = classifier_data_file or os.path.join(folder_path, "ObligationClassificationDataset.json")

        self._check_and_train_classifier()
        self.repass_scorer = RePASsScorer(self.classifier_path)
        self.linguistic_scorer = LinguisticScorer()

    def _model_exists(self):
        bin_file = os.path.join(self.classifier_path, "pytorch_model.bin")
        safe_file = os.path.join(self.classifier_path, "model.safetensors")
        return os.path.isfile(bin_file) or os.path.isfile(safe_file)

    def _check_and_train_classifier(self):
        if not self._model_exists():
            print("⚠️ Obligation classifier model not found. Training now...")
            trainer = ObligationClassifierTrainer(
                data_path=self.classifier_data_file,
                save_path=self.classifier_path
            )
            trainer.train()
        else:
            print("✅ Obligation classifier model is available.")

    def _harmonic_mean(self, a, b):
        return round((2 * a * b) / (a + b), 5) if (a + b) > 0 else 0.0

    def _repass_l_additive(self, entailment, contradiction, obligation_coverage, linguistic_score):
      """
      Additive RePASs-L formula:
      RePASs-L = (Entailment - Contradiction + ObligationCoverage + Linguistic + 1) / 4
      """
      return round((entailment - contradiction + obligation_coverage + linguistic_score + 1) / 4, 5)

    def _format_scores(self, scores_dict):
        return {k: round(float(v), 5) for k, v in scores_dict.items()}

    def run(self):
        with open(self.input_file, 'r', encoding='utf-8') as f:
            data = json.load(f)

        print(f"🚀 Running RePASs-L on {len(data)} items")
        results = []

        for idx, item in enumerate(data):
            qid = item["QuestionID"]
            retrieved = item.get("RetrievedPassages", item.get("RetrievedPassage(s)"))
            passage = " ".join(retrieved) if isinstance(retrieved, list) else retrieved

            answer = item["Answer"]

            # === Skip if Answer is None or empty ===
            if not answer or not answer.strip():
                print(f"[{idx + 1}/{len(data)}] Skipped QID: {qid} (empty or null answer)")
                continue

            repass_scores = self.repass_scorer.score(passage, answer)
            ling_scores = self.linguistic_scorer.score(passage, answer)

            repass_l_harmonic = self._harmonic_mean(
                repass_scores["repass_score"],
                ling_scores["combined_linguistic_score"]
            )

            repass_l_additive = self._repass_l_additive(
                repass_scores["entailment_score"],
                repass_scores["contradiction_score"],
                repass_scores["obligation_coverage_score"],
                ling_scores["combined_linguistic_score"]
            )

            results.append({
                "QuestionID": qid,
                **repass_scores,
                **ling_scores,
                "repass_l_score_additive": repass_l_additive,
                "repass_l_score_harmonic": repass_l_harmonic
            })

            print(
                f"[{idx + 1}/{len(data)}] Processed QID: {qid} → "
                f"RePASs: {self._format_scores(repass_scores)} | "
                f"Linguistic: {self._format_scores(ling_scores)} | "
                f"RePASs-L (Additive): {repass_l_additive:.5f}, Harmonic: {repass_l_harmonic:.5f}"
            )

        self._save_to_csv(results)
        self._save_summary(results)
        print(f"\n✅ Done. CSV and summary saved to: {os.path.dirname(self.output_file)}")

    def _save_to_csv(self, results):
        os.makedirs(os.path.dirname(self.output_file), exist_ok=True)
        with open(self.output_file, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=results[0].keys())
            writer.writeheader()
            writer.writerows(results)

    def _save_summary(self, results):
        df = pd.DataFrame(results)
        numeric_cols = df.select_dtypes(include='number')
        means = numeric_cols.mean()

        summary_path = os.path.join(os.path.dirname(self.output_file), "summary.txt")
        with open(summary_path, 'w', encoding='utf-8') as f:
            f.write("SUMMARY OF AVERAGE SCORES\n")
            f.write("==========================\n\n")
            f.write(f"Source: {os.path.basename(self.output_file)}\n\n")
            for col, avg in means.items():
                f.write(f"{col}: {round(avg, 5)}\n")

        print("\n=== 📊 Summary of Average Scores ===")
        for col, avg in means.items():
            print(f"{col}: {round(avg, 5)}")


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
## Import up sound alert dependencies
from IPython.display import Audio, display

def allDone():
  display(Audio(url='https://sound.peal.io/ps/audios/000/000/537/original/woo_vu_luvub_dub_dub.wav', autoplay=True))
## Insert whatever audio file you want above

In [ ]:
import nltk
nltk.download('punkt_tab')
import csv

if __name__ == "__main__":
    # === Configuration ===
    method_name = 'SAMPLE'
    folder_path = ""
    method_output_dir = os.path.join(folder_path, method_name)
    os.makedirs(method_output_dir, exist_ok=True)


    #input_file = os.path.join(folder_path, "sample2.json")
    input_file = "sample.json"
    output_file = os.path.join(method_output_dir, "repass_l_results.csv")

    # === Run RePASs-L Evaluation (Classifier check is done inside) ===
    runner = RePASSLRunner(
        folder_path=folder_path,
        input_file=input_file,
        output_file=output_file,
        classifier_data_file=os.path.join(folder_path, "ObligationClassificationDataset.json")
    )

    runner.run()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


✅ Obligation classifier model is available.


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


🚀 Running RePASs-L on 445 items
[1/445] Processed QID: 4d07cc7b-5fa2-4bed-a7e6-5f9e57e7ba4f → RePASs: {'entailment_score': 0.66883, 'contradiction_score': 0.00478, 'obligation_coverage_score': 0.0, 'repass_score': 0.55468} | Linguistic: {'fluency_score': 0.48878, 'readability_score': 0.01035, 'simplicity_score': 0.13012, 'combined_linguistic_score': 0.1019} | RePASs-L (Additive): 0.44149, Harmonic: 0.17217
[2/445] Skipped QID: 3ecd79c5-0dd2-4feb-8f53-6240a40d01bc (empty or null answer)
[3/445] Processed QID: 2b526fdc-4c41-4a8d-aa4d-68984e9fbe37 → RePASs: {'entailment_score': 0.0258, 'contradiction_score': 0.01943, 'obligation_coverage_score': 0.14286, 'repass_score': 0.38308} | Linguistic: {'fluency_score': 0.28669, 'readability_score': 0.2224, 'simplicity_score': 0.07813, 'combined_linguistic_score': 0.15579} | RePASs-L (Additive): 0.32625, Harmonic: 0.22150
[4/445] Processed QID: ee9a9935-98df-4b41-bbe2-057adaedc320 → RePASs: {'entailment_score': 0.26932, 'contradiction_score': 0.122

Token indices sequence length is longer than the specified maximum sequence length for this model (1307 > 1024). Running this sequence through the model will result in indexing errors


[28/445] Processed QID: 5bf8d233-9f27-4ed6-bf78-d341276adeb7 → RePASs: {'entailment_score': 0.55404, 'contradiction_score': 0.04826, 'obligation_coverage_score': 0.0, 'repass_score': 0.50193} | Linguistic: {'fluency_score': 1.0, 'readability_score': 0.21733, 'simplicity_score': 0.49382, 'combined_linguistic_score': 0.30661} | RePASs-L (Additive): 0.45310, Harmonic: 0.38068
[29/445] Skipped QID: 3ccb718d-a7eb-4e1c-979c-79baffa49101 (empty or null answer)
[30/445] Processed QID: cb613e70-17ee-4c90-b351-293dc22e5b3e → RePASs: {'entailment_score': 0.9241, 'contradiction_score': 0.05761, 'obligation_coverage_score': 0.125, 'repass_score': 0.66383} | Linguistic: {'fluency_score': 0.88025, 'readability_score': 0.16676, 'simplicity_score': 0.27499, 'combined_linguistic_score': 0.21093} | RePASs-L (Additive): 0.55060, Harmonic: 0.32014
[31/445] Skipped QID: a83a8d97-ab1f-48d5-b713-d5c76467c5d7 (empty or null answer)
[32/445] Processed QID: 82396265-fa7b-406d-8bfb-3d9589902383 → RePASs: {'entail

In [ ]:
allDone()